In [3]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from utils_transformer import TransformerEncoderWithAttention
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

In [4]:
scaler_X = MinMaxScaler()
dfs = pd.read_csv("../data/features.csv", index_col="Time_[s]")
targets = pd.read_csv("../data/targets.csv", index_col=False)
# df = dfs[dfs["Experiment_ID"]==2].drop(columns=["Experiment_ID"])
# target = targets[targets["Experiment_ID"]==2].iloc[:, 2:]
col = "Angle[degree]ORDistance[mm]"

scaler = MinMaxScaler(feature_range=(0, 1))
targets[col] = scaler.fit_transform(targets[[col]])
targets

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.317461,0.337050,0.626034,0.344543
1,2,0.022017,0.319825,0.363018,0.600159,0.372211
2,2,0.044033,0.319035,0.401244,0.560835,0.412939
3,2,0.066050,0.317768,0.438220,0.522651,0.452336
4,2,0.088067,0.320230,0.465094,0.495879,0.480970
...,...,...,...,...,...,...
14558,318,0.902686,0.641596,0.373375,0.680266,0.314326
14559,318,0.924703,0.640550,0.372634,0.680719,0.313536
14560,318,0.946720,0.640334,0.374073,0.679194,0.315070
14561,318,0.968736,0.641037,0.377992,0.675413,0.319245


In [5]:
groups_y = []
max_len_y = targets.groupby("Experiment_ID").size().max()
num_target_features = targets.shape[1] - 1  # exclude Experiment_ID

for exp_id, group in targets.groupby("Experiment_ID"):
    values = group.drop(columns=["Experiment_ID"]).values  # (len_group, num_target_features)
    
    # Pad with NaN
    padded = np.full((max_len_y, num_target_features), np.nan)
    padded[:values.shape[0], :] = values
    groups_y.append(padded)

# Final y: (num_experiments, max_len, num_target_features)
y = np.array(groups_y, dtype=float)

# Replace NaN with 0
y = np.nan_to_num(y, nan=0.0)



groups = []
max_len = dfs.groupby("Experiment_ID").size().max()  # longest experiment
num_features = dfs.shape[1] - 1  # exclude Experiment_ID

for exp_id, group in dfs.groupby("Experiment_ID"):
    features = group.drop(columns=["Experiment_ID"]).values
    
    # Pad with NaN (or zeros) to match max_len
    padded = np.full((max_len, num_features), np.nan)  
    padded[:features.shape[0], :] = features
    groups.append(padded)

X = np.array(groups, dtype=float)
X = np.nan_to_num(X, nan=0.0)
y = np.nan_to_num(y, nan=0.0)
print("Shape:", X.shape, y.shape)  
# Convert to tensors
X = torch.FloatTensor(X)
y = torch.FloatTensor(y)

Shape: (315, 1743, 17) (315, 47, 5)


In [6]:
# ---------------------------
# 1. Synthetic Data (Replace with your data)
# ---------------------------
num_samples = 315
seq_in = 1743
input_dim = 17
seq_out = 47
output_dim = 5


# ---------------------------
# 3. Training Setup
# ---------------------------
model = TransformerEncoderWithAttention()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---------------------------
# 4. Training Loop (Simple)
# ---------------------------
epochs = 1  # increase for real training
batch_size = 8

for epoch in range(epochs):
    perm = torch.randperm(num_samples)
    epoch_loss = 0
    for i in range(0, num_samples, batch_size):
        idx = perm[i:i+batch_size]
        x_batch, y_batch = X[idx], y[idx]

        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/(num_samples//batch_size):.4f}")




Epoch 1/1, Loss: 0.1981


In [7]:
# ---------------------------
# 5. Interpretability: Attention Heatmap for a Chosen Angle
# ---------------------------
# Pick a sample and angle
sample_idx = 0
angle_idx = 25  # 0-46
x_sample = X[sample_idx:sample_idx+1]  # (1, seq_in, input_dim)

model.eval()
with torch.no_grad():
    y_pred = model(x_sample)  # (1, seq_out, output_dim)

# Aggregate attention weights
# Note: self.attn.attn_output_weights is head x seq x seq, sum over heads
attn_sum = None
for att in model.attentions:
    if att.numel() == 1:  # skip empty
        continue
    att_avg = att.mean(dim=0).mean(dim=0)  # average over heads and batch
    attn_sum = att_avg if attn_sum is None else attn_sum + att_avg

if attn_sum is not None:
    plt.figure(figsize=(12,4))
    plt.plot(attn_sum.cpu().numpy())
    plt.title(f"Attention-based Importance for Angle {angle_idx}°")
    plt.xlabel("Timestamp")
    plt.ylabel("Importance")
    plt.show()
else:
    print("Attention weights not available for visualization. Consider using a modified Transformer layer to expose them.")


Attention weights not available for visualization. Consider using a modified Transformer layer to expose them.


In [6]:
model.attentions

[tensor([0.]), tensor([0.]), tensor([0.])]